# Linly-Dubbing Kaggle WebUI (v4 - Enhanced Error Handling)

This notebook is optimized for running **Linly-Dubbing** on Kaggle with **Dual T4 GPUs**. It includes advanced fixes for common Kaggle environment issues like `pynini` build failures and submodule import errors.

### 🚀 Key Improvements in v4
- **Robust Dependencies**: Fixed `pynini` and `whisperX` installation logic
- **Path Fallbacks**: Automatically adds submodules to `PYTHONPATH` if installation fails
- **Aggressive Caching**: Better handling of pre-installed packages
- **Python 3.12+ Ready**: Compatibility patches for new Kaggle images

### 📋 Execution Guide
1. **Step 1**: Clone repository and check GPU availability
2. **Step 2**: Install all dependencies (Uses robust multi-stage install)
3. **Step 3**: Download required AI models
4. **Step 4**: Launch the WebUI

### ⚙️ Kaggle Setup Requirements
- Enable **GPU T4 x2** in Settings → Accelerator
- Enable **Internet** in Settings → Internet

---

In [ ]:
# ============================================================================
# Step 0: 环境检测 (Environment Detection)
# ============================================================================

import os
import sys
import platform
import subprocess
import shutil

print("=" * 60)
print("🔍 Kaggle 运行环境检测")
print("=" * 60)

# Python 信息
print("\n📊 Python 环境:")
print(f"   Python 版本: {sys.version.split()[0]}")
print(f"   Python 路径: {sys.executable}")

# CUDA 信息
print("\n🎮 CUDA 环境:")
try:
    import torch
    print(f"   PyTorch 版本: {torch.__version__}")
    print(f"   CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   CUDA 版本: {torch.version.cuda}")
        try:
            print(f"   cuDNN 版本: {torch.backends.cudnn.version()}")
        except: print("   cuDNN 版本: 获取失败")
except ImportError:
    print("   ⚠️ PyTorch 未安装")

# 磁盘空间
stat = shutil.disk_usage('/kaggle/working')
print(f"\n💾 磁盘可用空间: {stat.free / (1024**3):.1f} GB")

# 预装 Python 包 (Handle torchvision error gracefully)
print("\n📦 预装 Python 包 (关键):")
for pkg in ['torch', 'torchvision', 'numpy', 'pip']:
    try: 
        m = __import__(pkg)
        print(f"   ✅ {pkg}: {getattr(m, '__version__', 'OK')}")
    except Exception as e:
        print(f"   ⚠️ {pkg}: 导入失败 ({type(e).__name__})")

print("\n" + "=" * 60)
print("✅ 环境检测完成!")
print("=" * 60)

In [ ]:
# ============================================================================
# Step 1: Clone Repository and Check GPU
# ============================================================================

import os
import torch

print("=" * 60)
print("🔍 GPU Detection")
print("=" * 60)
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"✅ Found {gpu_count} GPU(s):")
    for i in range(gpu_count):
        print(f"   - GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("❌ No GPU detected! Please enable GPU T4 x2 in settings.")

print("\n" + "=" * 60)
print("📦 Cloning Repository")
print("=" * 60)

if not os.path.exists('/kaggle/working/Linly-Dubbing'):
    !cd /kaggle/working && git clone https://github.com/Kedreamix/Linly-Dubbing.git --depth 1
else:
    !cd /kaggle/working/Linly-Dubbing && git pull

%cd /kaggle/working/Linly-Dubbing

print("\nInitializing submodules...")
!git submodule update --init --recursive

print("\n✅ Step 1 Complete!")

In [ ]:
# ============================================================================
# Step 2: Install Dependencies (Robust Version for Kaggle)
# ============================================================================

print("=" * 60)
print("📦 Installing System Dependencies")
print("=" * 60)

# 1. Install system tools required for build
!apt-get update -qq
!apt-get install -y -qq build-essential libfst-dev libfst-tools python3-dev ffmpeg

print("\n🐍 Installing Python Dependencies")
print("=" * 60)

# 2. Update pip tools
!pip install --upgrade -q pip setuptools wheel

# 3. Applying patches to requirements.txt
!sed -i 's/numpy==1.26.3/numpy<2.0.0/g' requirements.txt
!sed -i '/pynini/d' requirements.txt

# 4. Patch TTS for Python 3.12 compatibility
if os.path.exists('submodules/TTS/setup.py'):
    !sed -i 's/Version(python_version) >= Version("3.12"): /Version(python_version) >= Version("3.13"): /g' submodules/TTS/setup.py
    !sed -i 's/python_requires=">=3.9.0, <3.12",/python_requires=">=3.9.0, <3.13",/g' submodules/TTS/setup.py

# 5. Robust pynini installation
print("Installing pynini...")
!pip install -q pynini==2.1.5 --no-cache-dir || (print("Retry default pynini...") && pip install -q pynini --no-cache-dir)

# 6. Install core requirements
print("Installing core requirements...")
!pip install -q -r requirements.txt

# 7. Install submodules with better handling
print("Installing submodules...")
submodules = ['submodules/demucs', 'submodules/whisper', 'submodules/whisperX', 'submodules/TTS']
for sm in submodules:
    if os.path.exists(sm):
        print(f"   - {sm}...")
        if 'whisperX' in sm: 
            !pip install -q pyannote.audio==3.1.1 faster-whisper==1.0.0
        !pip install -q -e {sm} --no-deps || print(f"   ⚠️ Editable install failed for {sm}, will use path fallback.")

# 8. Final tools
!pip install -q loguru yt-dlp gradio==4.44.1

print("\n✅ Step 2 Complete!")

In [ ]:
# ============================================================================
# Step 3: Download AI Models
# ============================================================================

print("=" * 60)
print("🤖 Downloading AI Models")
print("=" * 60)

!mkdir -p models/ASR/whisper
wav2vec_path = 'models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth'
if not os.path.exists(wav2vec_path):
    !wget -nc https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth -O {wav2vec_path}

# Use download script
!python scripts/huggingface_download.py

print("\n✅ Step 3 Complete!")

In [ ]:
# ============================================================================
# Step 4: Launch WebUI
# ============================================================================

import os
import sys

print("=" * 60)
print("🚀 Launching Linly-Dubbing WebUI")
print("=" * 60)

# Ensure submodules are in path (Fallback for installation issues)
project_root = '/kaggle/working/Linly-Dubbing'
submodule_paths = [
    os.path.join(project_root, 'submodules/demucs'),
    os.path.join(project_root, 'submodules/whisper'),
    os.path.join(project_root, 'submodules/whisperX'),
    os.path.join(project_root, 'submodules/TTS')
]
for p in submodule_paths:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

# Set environment variables
os.environ['PYTHONPATH'] = ':'.join(submodule_paths) + (':' + os.environ.get('PYTHONPATH', '') if os.environ.get('PYTHONPATH') else '')
os.environ['MPLBACKEND'] = 'Agg'

if not os.path.exists('.env'):
    !cp env.example .env

print("🌐 Starting Gradio WebUI...")
!python webui.py